# Data Preparation and Validation
This notebook builds instruction-format data from Hugging Face datasets and validates quality + leakage.

In [ ]:
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import yaml
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path('..').resolve()
CONFIG = yaml.safe_load((ROOT / 'config.yaml').read_text())
SEED = CONFIG['seed']
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# 1) Example loading from HF sources (reproducibility reference)
# You can uncomment to rebuild data from source datasets:
# ag = load_dataset('ag_news', split='train[:200]')
# emo = load_dataset('dair-ai/emotion', split='train[:200]')
# yelp = load_dataset('yelp_polarity', split='train[:200]')
# sms = load_dataset('sms_spam', split='train[:200]')

train_path = ROOT / CONFIG['dataset']['train_path']
test_path = ROOT / CONFIG['dataset']['test_path']

def read_jsonl(path: Path):
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

train_df = read_jsonl(train_path)
test_df = read_jsonl(test_path)
print('train:', len(train_df), 'test:', len(test_df))


In [ ]:
# 2) Validation checks + duplicate cleanup
schema = set(CONFIG['schema']['required_fields'])

def parse_output(text):
    try:
        obj = json.loads(text)
        return obj, None
    except Exception as e:
        return None, str(e)

def dedupe_df(df):
    before = len(df)
    deduped = df.drop_duplicates(subset=['instruction', 'input', 'output']).reset_index(drop=True)
    return deduped, before - len(deduped)

report = {
    'empty_values': 0,
    'invalid_json': 0,
    'too_long': 0,
    'train_duplicates_removed': 0,
    'test_duplicates_removed': 0,
    'cross_split_exact_overlap': 0,
}

train_df, report['train_duplicates_removed'] = dedupe_df(train_df)
test_df, report['test_duplicates_removed'] = dedupe_df(test_df)

train_keys = set(zip(train_df['instruction'], train_df['input'], train_df['output']))
test_keys = set(zip(test_df['instruction'], test_df['input'], test_df['output']))
report['cross_split_exact_overlap'] = len(train_keys.intersection(test_keys))

for df_name, df in [('train', train_df), ('test', test_df)]:
    for _, row in df.iterrows():
        if not str(row['instruction']).strip() or not str(row['input']).strip() or not str(row['output']).strip():
            report['empty_values'] += 1

        parsed, err = parse_output(row['output'])
        if err:
            report['invalid_json'] += 1
        elif set(parsed.keys()) != schema:
            report['invalid_json'] += 1

        if len(row['input']) > 2000 or len(row['output']) > 600:
            report['too_long'] += 1

print(report)
print('train after dedupe:', len(train_df), 'test after dedupe:', len(test_df))


In [ ]:
# 3) Subtype distribution

def subtype_distribution(df):
    labels = []
    for txt in df['output'].tolist():
        labels.append(json.loads(txt)['task_type'])
    return Counter(labels)

print('Train subtype distribution:', subtype_distribution(train_df))
print('Test subtype distribution:', subtype_distribution(test_df))


In [ ]:
# 4) Leakage check via embedding cosine similarity + mitigation + persist files
model_name = CONFIG['leakage']['embedding_model']
threshold = CONFIG['leakage']['similarity_threshold']
embedder = SentenceTransformer(model_name)

train_inputs = train_df['input'].astype(str).tolist()
test_inputs = test_df['input'].astype(str).tolist()

train_emb = embedder.encode(train_inputs, convert_to_numpy=True, normalize_embeddings=True)
test_emb = embedder.encode(test_inputs, convert_to_numpy=True, normalize_embeddings=True)

sim = cosine_similarity(test_emb, train_emb)
leaks = []
for i in range(sim.shape[0]):
    j = int(sim[i].argmax())
    score = float(sim[i, j])
    if score >= threshold:
        leaks.append((i, j, score, test_inputs[i], train_inputs[j]))

print(f'Potential leakage pairs: {len(leaks)}')

clean_test_df = test_df.copy()
if leaks:
    leak_df = pd.DataFrame(leaks, columns=['test_idx', 'train_idx', 'score', 'test_input', 'train_input'])
    display(leak_df.head(10))

    leaked_test_idx = sorted(set(leak_df['test_idx'].tolist()))
    clean_test_df = test_df.drop(index=leaked_test_idx).reset_index(drop=True)

    print('Action taken: removed leaked samples from test split')
    print('Removed from test:', len(leaked_test_idx))
    print('New test size:', len(clean_test_df))
else:
    print('No leakage above threshold. No split changes required.')

if len(clean_test_df) < CONFIG['dataset']['min_test_samples']:
    raise ValueError('Test split became too small after leakage cleanup. Regenerate dataset split.')

def write_jsonl(df, path):
    with path.open('w', encoding='utf-8') as f:
        for row in df.to_dict(orient='records'):
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

write_jsonl(train_df, train_path)
write_jsonl(clean_test_df, test_path)
print('Saved cleaned train/test files to disk.')
